# Session 2B: RAG-Powered Agent
**Goal**: Give the agent semantic memory so it reads your rules BEFORE scheduling.

In [ ]:
import os
import sys
sys.path.insert(0, '..')
import utils.bootstrap

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_chroma import Chroma
from langgraph.prebuilt import create_react_agent
from utils.tools import fetch_emails, schedule_meeting
from utils.llm_router import get_routed_llm

### Load Vector Store (RAG)

In [ ]:
CHROMA_DIR = os.path.abspath('.chroma_db')
embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')
vectorstore = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)

@tool
def search_user_preferences(query: str) -> str:
    """Search the user's personal preferences and rules."""
    docs = vectorstore.similarity_search(query, k=3)
    return '\n'.join([d.page_content for d in docs]) if docs else 'No rules found.'

### Run the Agent

In [ ]:
llm = get_routed_llm(role='worker_model')
tools = [fetch_emails, search_user_preferences, schedule_meeting]

system_msg = SystemMessage(content="ALWAYS search preferences BEFORE scheduling. Refuse and suggest alternatives if it violates rules.")
agent = create_react_agent(llm, tools)

print("\ud83e\udd16 Running RAG Agent...")
result = agent.invoke({"messages": [system_msg, HumanMessage(content="Fetch emails. Check my preferences before scheduling any meetings.")]})

for msg in result['messages']:
    print(f"\n[{msg.type.upper()}]: {str(msg.content)[:300]}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  \ud83d\udd27 Tool Call: {tc['name']}({tc['args']})")